# Gradient descent and PyTorch multislice

## Major steps
1. Use abTEM as an initialization step, take the user-provided params (i.e., max thickness, unit cell) to generate potential and scan position (Ang)
2. Preprocess the potential into object phase shift, and scan position into pixels
3. Prepare the probe and multislice propagator (Need to be careful about the probe size at the end of the slice)
4. Initialize the AD model with optimizers and params. Probe and position are fixed, and prepare the object with a differentiable soft mask
5. Feed the scan position, object, probe, propagator into the multislice forward function to get diffraction pattern
6. Make PACBED out of all diffraction patterns 
7. Calculate the loss (We might be able to add reduction methods here)
8. Backpropagate the grad
9. Take the optimizer step to update the params (thickness, tilt_x, tilt_y) for 1 update step

## Notes
- Use broadcasting to process tilts and thicknesses in parallel
- Have an option to output all the CBED from intermediate thickness
  - This option would probably be used for qBO, not the GD baseline

In [ ]:
import os
work_dir = "H:/workspace/bott"
os.chdir(work_dir)
print("Current working dir: ", os.getcwd())

In [ ]:
from scipy.ndimage import zoom

import torch
from torch.fft import fft2, ifft2, fftfreq

In [ ]:
def fftshift2(x):
    """ A wrapper over torch.fft.fftshift for the last 2 dims """
    # Note that fftshift and ifftshift are only equivalent when N = even 
    return torch.fft.fftshift(x, dim=(-2,-1))  

def ifftshift2(x):
    """ A wrapper over torch.fft.ifftshift for the last 2 dims"""
    # Note that fftshift and ifftshift are only equivalent when N = even 
    return torch.fft.ifftshift(x, dim=(-2,-1))  

In [ ]:
def scaled_sigmoid(length, offset=0, scale=1, device='cuda'):
    """
    Generate a scaled sigmoid decay over `length` positions, optionally with per-sample offsets.

    Parameters:
        length (int): Number of positions along x-axis.
        offset (float or 1D tensor): Center location(s) where the sigmoid is 0.5.
        scale (float): Controls steepness of decay. Higher means slower drop.

    Returns:
        Tensor of shape (length,) if offset is scalar, or (N, length) if offset is 1D tensor of size N.
    """
    # If scale =  1, y drops from 1 to 0 between (-0.5,0.5), or effectively 1 px
    # If scale = 10, it takes roughly 10 px for y to drop from 1 to 0
    
    x = torch.arange(length, device=device)

    if torch.is_tensor(offset):
        x = x.view(1, -1)  # shape (1, length)
        offset = offset.view(-1, 1)  # shape (N, 1)
    scaled_sigmoid = 1 / (1 + torch.exp((x - offset) / scale * 10))

    return scaled_sigmoid

In [ ]:
def multislice_forward_model_vec_all(probe, H, tmask, objp_patches, eps=1e-10):
    """
    Computes the multislice position-averaged electron diffraction pattern (PACBED) 
    with AD-optimizable thickness and tilts using probe and mixed-state object
    using a fully vectorized forward model.

    Args:
        probe (torch.Tensor): Tensor of shape (N, Ny, Nx) with complex64 values, 
            representing the probe(s). N is the number of samples in the batch, pmode is the 
            number of probe modes. By default, N is 1, assuming the same probe for all samples.
        H (torch.Tensor): Tensor of shape (num_tilts Ky, Kx) with complex64 values, representing the Fresnel 
            propagator that propagates the wave by a slice thickness given a tilt.
        tmask (torch.Tensor): Tensor of shape (num_thickness, Nz) with float32 values, representing
            the sigmoid mask used to zero out the contribution from object slices beyond the target slice (offset) 
        objp_patches (torch.Tensor): Tensor of shape (N, omode, Nz, Ny, Nx), representing
            object phase patches with float32. 
            N is the number of samples in a batch, omode is the number of object modes, 
            Nz, Ny, Nx are the dimensions of the object patches.
        eps (float, optional): A small value added for numerical stability. Defaults to 1e-10.

    Returns:
        torch.Tensor: Tensor of shape (num_tilts, num_thickness, Ky, Kx) with float32 positive values, representing the 
        forward PACBED pattern for each tilt and thickness. The PACBED is generated by averaging all DP from each probe position specified by the batch indices.
    """
    
    # Multiply object phase with the thickness mask
    objp_patches = objp_patches[:,None] * tmask[...,None,:,None,None] # objc_patches (N, num_thickness, omode, Nz, Ny, Nx)
    
    # Cast the object back to actual complex tensor
    object_cplx = torch.exp(1j*objp_patches)
    n_slices = object_cplx.shape[-3]

    # Expand psi to include num_tilts, num_thickness, and omode dimension
    psi = probe[:, None, None, None, :, :] # (N, Ny, Nx) -> (N, 1, 1, 1, Ny, Nx) -> (N, num_tilts, num_thickness, omode, Ny, Nx)

    # Propagating each object layer using broadcasting
    for n in range(n_slices-1):
        object_slice = object_cplx[..., n, :, :]  # object_slice -> (N, num_thickness, omode, Ny, Nx)
        psi = psi * object_slice[:, None] # psi -> (N, num_tilts, num_thickness, omode, Ny, Nx). Note that psi is always centered in real space
        psi = ifft2(H[:,None,None] * fft2(psi))    # H -> (num_tilts, 1, 1, Ny, Nx). Note that fft2 and ifft2 are applying to the last 2 axes. 

    # Interacting with the last layer, and no propagation is needed afterward
    object_slice = object_cplx[..., n_slices-1, :, :]
    psi = psi * object_slice[:, None]

    # Propagate the object-modified exit wave psi(r) to detector plane into psi(k)
    # The batch and omode dimension are averaged to generate the PACBED
    # dp_fwd (num_tilts, num_thickness, Ny, Nx)
    dp_fwd = (
        torch.mean(
            (fftshift2(fft2(psi, norm="ortho"))).abs().square(),
            dim=(0,-3),
        )
        + eps
    )
    return dp_fwd

In [ ]:
def near_field_evolution_torch(Npix_shape, dx, dz, lambd, device='cuda'):
    """Fresnel propagator in PyTorch"""
    # This is for testing and demonstration purpose and is never called in PtyRAD
    # The actual optimizable Fresnel propagator is directly built inside "PtychoAD.get_propagators"
    dx    = dx.to(device)
    dz    = dz.to(device)
    lambd = lambd.to(device)

    ygrid = (torch.arange(-Npix_shape[-2] // 2, Npix_shape[-2] // 2, device=device) + 0.5) / Npix_shape[-2]
    xgrid = (torch.arange(-Npix_shape[-1] // 2, Npix_shape[-1] // 2, device=device) + 0.5) / Npix_shape[-1]

    # Standard ASM
    k  = 2 * torch.pi / lambd
    ky = 2 * torch.pi * ygrid / dx
    kx = 2 * torch.pi * xgrid / dx
    Ky, Kx = torch.meshgrid(ky, kx, indexing="ij")
    H = torch.fft.ifftshift(torch.exp(1j * dz * torch.sqrt(k ** 2 - Kx ** 2 - Ky ** 2))) # H has zero frequency at the corner in k-space

    return H

In [ ]:
def make_stem_probe(params_dict, verbose=True):
    # MAKE_TEM_PROBE Generate probe functions produced by object lens in 
    # transmission electron microscope.
    # Written by Yi Jiang based on Eq.(2.10) in Advanced Computing in Electron 
    # Microscopy (2nd edition) by Dr.Kirkland
    # Implemented and slightly modified in python by Chia-Hao Lee
 
    # Outputs:
        #  probe: complex probe functions at real space (sample plane)
    # Inputs: 
        #  params_dict: probe parameters and other settings
    
    import numpy as np
    from numpy.fft import fftfreq, fftshift, ifft2, ifftshift
    
    ## Basic params
    voltage     = float(params_dict["kv"])         # Ang
    conv_angle  = float(params_dict["conv_angle"]) # mrad
    Npix        = int  (params_dict["Npix"])       # Number of pixel of thr detector/probe
    dx          = float(params_dict["dx"])         # px size in Angstrom
    ## Aberration coefficients
    df          = float(params_dict["df"]) #first-order aberration (defocus) in angstrom
    c3          = float(params_dict["c3"]) #third-order spherical aberration in angstrom
    c5          = float(params_dict["c5"]) #fifth-order spherical aberration in angstrom
    c7          = float(params_dict["c7"]) #seventh-order spherical aberration in angstrom
    f_a2        = float(params_dict["f_a2"]) #twofold astigmatism in angstrom
    f_a3        = float(params_dict["f_a3"]) #threefold astigmatism in angstrom
    f_c3        = float(params_dict["f_c3"]) #coma in angstrom
    theta_a2    = float(params_dict["theta_a2"]) #azimuthal orientation in radian
    theta_a3    = float(params_dict["theta_a3"]) #azimuthal orientation in radian
    theta_c3    = float(params_dict["theta_c3"]) #azimuthal orientation in radian
    shifts      = params_dict["shifts"] #shift probe center in angstrom
    
    # Calculate some variables
    wavelength = 12.398/np.sqrt((2*511.0+voltage)*voltage) #angstrom
    k_cutoff = conv_angle/1e3/wavelength
    dk = 1/(dx*Npix)
    
    if verbose:
        print("Start simulating STEM probe")
    
    # Make k space sampling and probe forming aperture
    kx = fftshift(fftfreq(Npix, 1/Npix))
    # kx = np.linspace(-np.floor(Npix/2),np.ceil(Npix/2)-1,Npix)
    kX,kY = np.meshgrid(kx,kx, indexing='xy')

    kX = kX*dk
    kY = kY*dk
    kR = np.sqrt(kX**2+kY**2)
    theta = np.arctan2(kY,kX)
    mask = (kR<=k_cutoff).astype('bool') 
    
    # Adding aberration one-by-one, the aberrations modify the flat phase (imagine a flat wavefront at aperture plane) with some polynomial perturbations
    # The aberrated phase is called chi(k), probe forming aperture is placed here to select the relatively flat phase region to form desired real space probe
    # Note that chi(k) is real-valued function with unit as radian, it's also not limited between -pi,pi. Think of phase shift as time delay might help.
    
    chi = -np.pi*wavelength*kR**2*df
    if c3!=0: 
        chi += np.pi/2*c3*wavelength**3*kR**4
    if c5!=0: 
        chi += np.pi/3*c5*wavelength**5*kR**6
    if c7!=0: 
        chi += np.pi/4*c7*wavelength**7*kR**8
    if f_a2!=0: 
        chi += np.pi*f_a2*wavelength*kR**2*np.sin(2*(theta-theta_a2))
    if f_a3!=0: 
        chi += 2*np.pi/3*f_a3*wavelength**2*kR**3*np.sin(3*(theta-theta_a3))
    if f_c3!=0: 
        chi += 2*np.pi/3*f_c3*wavelength**2*kR**3*np.sin(theta-theta_c3)

    psi = np.exp(-1j*chi)*np.exp(-2*np.pi*1j*shifts[0]*kX)*np.exp(-2*np.pi*1j*shifts[1]*kY)
    probe = mask*psi # It's now the masked wave function at the aperture plane
    probe = fftshift(ifft2(ifftshift(probe))) # Propagate the wave function from aperture to the sample plane. 
    probe = probe/np.sqrt(np.sum((np.abs(probe))**2)) # Normalize the probe so sum(abs(probe)^2) = 1

    if verbose:
        # Print some useful values
        print(f'kv          = {voltage} kV')    
        print(f'wavelength  = {wavelength:.4f} Ang')
        print(f'conv_angle  = {conv_angle} mrad')
        print(f'Npix        = {Npix} px')
        print(f'dk          = {dk:.4f} Ang^-1')
        print(f'kMax        = {(Npix*dk/2):.4f} Ang^-1')
        print(f'alpha_max   = {(Npix*dk/2*wavelength*1000):.4f} mrad')
        print(f'dx          = {dx:.4f} Ang, Nyquist-limited dmin = 2*dx = {2*dx:.4f} Ang')
        print(f'Rayleigh-limited resolution  = {(0.61*wavelength/conv_angle*1e3):.4f} Ang (0.61*lambda/alpha for focused probe )')
        print(f'Real space probe extent = {dx*Npix:.4f} Ang')
    
    return probe

In [ ]:
def imshift_batch(img, shifts, grid):
    """
    Generates a batch of shifted images from a single input image (..., Ny,Nx) with arbitray leading dimensions.
    
    This function shifts a complex/real-valued input image by applying phase shifts in the Fourier domain,
    achieving subpixel shifts in both x and y directions.

    Inputs:
        img (torch.Tensor): The input image to be shifted. 
                            img could be either a mixed-state complex probe (pmode, Ny, Nx) complex64 tensor, 
                            or a mixed-state pseudo-complex object stack (2,omode,Nz,Ny,Nx) float32 tensor.
        shifts (torch.Tensor): The shifts to be applied to the image. It should be a (Nb,2) tensor and each slice as (shift_y, shift_x).
        grid (torch.Tensor): The k-space grid used for computing the shifts in the Fourier domain. It should be a tensor with shape=(2, Ny, Nx),
                             where Ny and Nx are the height and width of the images, respectively. Note that the grid is normalized so the value spans
                             from 0 to 1

    Outputs:
        shifted_img (torch.Tensor): The batch of shifted images. It has an extra dimension than the input image, i.e., shape=(Nb, ..., Ny, Nx),
                                    where Nb is the number of samples in the input batch.

    Note:
        - The shifts are in unit of pixel. For example, a shift of (0.5, 0.5) will shift the image by half a pixel in both y and x directions, positive is down/right-ward.
        - The function utilizes the fast Fourier transform (FFT) to perform the shifting operation efficiently.
        - Make sure to convert the input image and shifts tensor to the desired device before passing them to this function.
        - The fft2 and fftshifts are all applied on the last 2 dimensions, therefore it's only shifting along y and x directions
        - tensor[None, ...] would add an extra dimension at 0, so *[None]*ndim means unwrapping a list of ndim None as [None, None, ...]
        - The img is automatically broadcast to (Nb, *img.shape), so if a batch of images are passed in, each image would be shifted independently
    """
    
    assert img.shape[-2:] == grid.shape[-2:], f"Found incompatible dimensions. img.shape[-2:] = {img.shape[-2:]} while grid.shape[-2:] = {grid.shape[-2:]}"
    
    ndim = img.ndim                                                                   # Get the total img ndim so that the shift is dimension-indepent
    shifts = shifts[(...,) + (None,) * ndim]                                          # Expand shifts to (Nb,2,1,1,...) so shifts.ndim = ndim+2. It was written as `shifts = shifts[..., *[None]*ndim]` for Python 3.11 or above with better readability
    grid = grid[(slice(None),) + (None,) * (ndim - 1) + (...,)]                       # Expand grid to (2,1,1,...,Ny,Nx) so grid.ndim = ndim+2. It was written as `grid = grid[:,*[None]*(ndim-1), ...]` for Python 3.11 or above with better readability
    shift_y, shift_x = shifts[:, 0], shifts[:, 1]                                     # shift_y, shift_x are (Nb,1,1,...) with ndim singletons, so the shift_y.ndim = ndim+1
    ky, kx = grid[0], grid[1]                                                         # ky, kx are (1,1,...,Ny,Nx) with ndim-2 singletons, so the ky.ndim = ndim+1
    w = torch.exp(-(2j * torch.pi) * (shift_x * kx + shift_y * ky))                   # w = (Nb, 1,1,...,Ny,Nx) so w.ndim = ndim+1. w is at the center.
    shifted_img = ifft2(ifftshift2(fftshift2(fft2(img)) * w))                         # For real-valued input, take shifted_img.real
    
    # Note that for imshift, it's better to keep fft2(img) than fft2(ifftshift2(img))
    # While fft2(img).angle() might seem serrated, it's indeed better to keep it as is, which is essentially setting the center as the origin for FFT.
    
    return shifted_img

## Configure abTEM parameter for the ground truth

In [ ]:
# Input unknown
thickness, tilt_x, tilt_y = 100, 20, -20 #(Ang, mrad, mrad)

# Setup abtem device
device_abtem = 'gpu'

# Read cif into abTEM for 4D-STEM generation
path_crystal = './data/SrTiO3.cif'

# Potential params
potential_extent_x = 62.6 # Ang
potential_extent_y = 62.6 # Ang
lateral_sampling = 0.2*2/3 # Ptycho recon pixel size = 0.2 Ang, so ptycho CBED need 1/2dx = 2.5 Ang-1 kMax. Considering the 2/3 kMax antialias, we need 2.5*1.5 = 3.75 Ang-1 for abTEM CBED or equivalently 0.1333 Ang px
vertical_sampling = 2
potential_parametrization = "lobato" # Seems to be more accurate then "kirkland"
potential_projection = "finite" # infinite is faster but less accurate
exit_planes = None # Output diffraction pattern every N slices

# Phonon params
random_seed = 42
use_frozen_phonon = False
num_phonon_configs = 5
phonon_sigma = {'Sr':.088,'Ti':.0746,'O':.0963} #0.1 # Ang 

# Probe params
energy = 200e3
convergence_angle = 19.1
df = 0
aberrations = {}

# Scan and pacbed
scan_step_size = 0.3
return_pacbed = False

## Everything below is just abTEM calculation

In [ ]:
# Setup imports
from bott.utils import get_EM_constants

import ase
import abtem
import numpy as np
import dask

if device_abtem == 'gpu':
    import cupy as xp
    abtem.config.set({"dask.chunk-size-gpu" : "2048 MB"})
    dask.config.set({"num_workers": 1});
elif device_abtem == 'cpu':
    import numpy as xp
else:
    raise ValueError(f"device_abtem '{device_abtem}' not implemented yet, please use 'cpu', or 'gpu'!")

# abtem configure
abtem.config.set({"local_diagnostics.progress_bar": False});
abtem.config.set({"device": device_abtem});

import matplotlib.pyplot as plt

In [ ]:
# Setup cell
unit_cell = ase.io.read(path_crystal)
target_object_extent = np.array((potential_extent_x, potential_extent_y, thickness)) # Specimen range in Ang (x,y,z)
cell_constants = np.diag(unit_cell.cell)
super_cell_reps = np.ceil(target_object_extent / cell_constants).astype('int')
super_cell = unit_cell * super_cell_reps
print(f"super_cell = {super_cell}")

In [ ]:
# Calculate the potential
if use_frozen_phonon:
    print(f"Using FrozenPhonons potential with {num_phonon_configs} configs")
    atoms = abtem.FrozenPhonons(atoms=super_cell, num_configs=num_phonon_configs, sigmas=phonon_sigma, seed=random_seed)
else:
    print("Using Static potential")
    atoms = super_cell

potential = abtem.Potential(atoms=atoms, sampling=lateral_sampling, parametrization=potential_parametrization,
        slice_thickness=vertical_sampling, projection=potential_projection, exit_planes=exit_planes)
potential_arr = potential.build().compute(progress_bar=False).array.transpose(0,2,1)
print("Note that the last 2 axes are transposed because abTEM go with (z,x,y) but we want (z,y,x)")

if device_abtem == 'gpu':
    potential_arr = potential_arr.get()

print(f"potential.shape = {potential.shape}")
print(f"potential_arr.shape = {potential_arr.shape}")

In [ ]:
# Calculate the probe
probe = abtem.Probe(energy=energy, semiangle_cutoff=convergence_angle, defocus=df, tilt=(tilt_x, tilt_y), **aberrations)
probe.grid.match(potential)
probe_arr = probe.build().compute().array

if device_abtem == 'gpu':
    probe_arr = probe_arr.get()

# Useful information
wavelength = get_EM_constants(energy/1e3, 'wavelength')
kmax_antialias = 1/lateral_sampling/3 # 1/Ang #The kmax_antialiasing = 2.675 Ang-1 
alpha_max_antialias = wavelength * kmax_antialias # rad
print(f"Energy = {energy/1e3} kV, rel. wavelength = {wavelength:.4g} Ang")
print(f"CBED collection kmax = {kmax_antialias:.4g} 1/Ang, collection alpha_max = {alpha_max_antialias*1000:.4g} mrad")
print(f"probe.shape = {probe.shape}")
print(f"probe.axes_metadata = {probe.axes_metadata}")

In [ ]:
############################################################
### Make scan positions, unit in Ang
############################################################

N_scan_fast = np.ceil(cell_constants[0] / scan_step_size).astype(int)
N_scan_slow = np.ceil(cell_constants[1] / scan_step_size).astype(int)

pos = scan_step_size * np.array([(y, x) for y in range(N_scan_slow) for x in range(N_scan_fast)]) # (N,2), each row is (y,x)
pos = pos - pos.mean(0) # Center scan around origin

# Apply offset to move the scan pattern inside the supercell
offset = pos.min(0) - 30
pos -= offset

# Parse the position into the hdf5 for reconstuction, and the abTEM scan position
pos_ang_yx = pos
pos_ang_xy = np.flip(pos,1)

print(f"N_scan_fast = {N_scan_fast}")
print(f"N_scan_slow = {N_scan_slow}")
print(f"pos_ang_xy[:5] = {pos_ang_xy[:5]}")

In [ ]:
# Get cbeds, remove the [0] from pos_ang_xy to make it scan the whole unit cell
cbeds = probe.multislice(scan = pos_ang_xy[0], potential = potential).diffraction_patterns(max_angle='cutoff').reduce_ensemble().compute().array

if return_pacbed:
    measurement_arr = xp.mean(cbeds, axis=(-3)) # Reduce the scan axes
else:
    measurement_arr = cbeds
    
if device_abtem == 'gpu':
    measurement_arr = measurement_arr.get()

In [ ]:
plt.figure()
plt.imshow(measurement_arr**0.5)
plt.show()

## Preprocess and Initialize the abTEM arrays for GD baseline

In [ ]:
# Setup the optimization parameters
Npix = 256 # This is the size of the CBED
dx = lateral_sampling * 3/2 # Ang

batch_size = 1 # Note that for a single mini-batch, the batch_size needs to be as large as possible to be a good approximation of the original PACBED.
indices = [0] #np.random.choice(np.arange(len(pos_ang_xy)), batch_size, replace=False)

# Note that the order and tilt_y, tilt_x and the directions are both reversed from abTEM to my own convention.
# I go with tilt_y, tilt_x, and my positive direction corresponds to abTEM's negative direction

# num_tilts = 1
# tilt_min, tilt_max = -20, 20
# tilts = torch.tensor(np.random.uniform(tilt_min, tilt_max, size=(num_tilts,2)), device='cuda', requires_grad=True)
tilts = -1*torch.tensor([[-20, 20]], device='cuda')

# num_thickness = 1
# thickness_min, thickness_max = 10, 25 # Ang
# thicknesses = torch.tensor(np.linspace(thickness_min, thickness_max, num_thickness, endpoint=True), device='cuda', requires_grad=True)
thicknesses = torch.tensor([110], device='cuda')

In [ ]:
# Initialize PACBED
scale_factors = [Npix/measurement_arr.shape[0], Npix/measurement_arr.shape[1]]
meas = zoom(measurement_arr, zoom=scale_factors, order=1) #/ np.prod(scale_factors) # Dividing by scale_factors would keep meas.sum() roughly conserved
meas /= meas.sum() # Normalize it such that the PACBED sum at 1
meas = meas.T # Apply transpose to match the orientation
print(f"meas.shape = {meas.shape}, meas.sum() = {meas.sum()}")

In [ ]:
# Initialize probe
probe_params = {
    "kv": energy/1e3,         # Ang
    "conv_angle": convergence_angle, # mrad
    "Npix": Npix,       # Number of pixel of thr detector/probe
    "dx": dx, # px size in Angstrom
    ## Aberration coefficients
    "df": df, #first-order aberration (defocus) in angstrom
    "c3": 0, #third-order spherical aberration in angstrom
    "c5": 0, #fifth-order spherical aberration in angstrom
    "c7":0, #seventh-order spherical aberration in angstrom
    "f_a2":0, #twofold astigmatism in angstrom
    "f_a3":0, #threefold astigmatism in angstrom
    "f_c3":0, #coma in angstrom
    "theta_a2":0, #azimuthal orientation in radian
    "theta_a3":0, #azimuthal orientation in radian
    "theta_c3":0, #azimuthal orientation in radian
    "shifts":[0,0], #shift probe center in angstrom
}
probe = make_stem_probe(probe_params) # The probe is by default intensity sum at 1
print(f"probe.shape = {probe.shape}, sum(probe.abs()**2) = {np.sum(np.abs(probe)**2)}")


In [ ]:
# Initialize object
scale_factors = [1]*(potential_arr.ndim-2) + [2/3] + [2/3] # Resample the pixel size to 3/2 because the kMax was crop to 2/3 by abTEM
potential = zoom(potential_arr, zoom=scale_factors, order=1) # The potential value is now resampled but no need to scale the value.
print(f"potential_arr.shape = {potential_arr.shape}, potential_arr.sum(-3).mean() = {potential_arr.sum(-3).mean()}")
print(f"potential.shape     = {potential.shape},     potential.sum(-3).mean() = {potential.sum(-3).mean()}")

objp = get_EM_constants(energy/1e3, 'sigma') * potential/1e3 # Convert potential to the phase of the complex object
if objp.ndim == 3:
    objp = objp[None,]
print(f"objp.shape           = {objp.shape},     objp.sum(-3).mean() = {objp.sum(-3).mean()}")

In [ ]:
# Preprocess the position so that it's compatible with follow up reconstruction packages
pos_px_yx = pos_ang_yx / dx
probe_shape = np.array(probe.shape[-2:])
pos_px_yx = pos_px_yx - probe_shape/2 # Shift back to obj coordinate 

crop_pos = np.round(pos_px_yx)
probe_pos_shifts = pos_px_yx - np.round(pos_px_yx)

# Visualize it in conventional orientation, although abTEM would put origin at the bottom left
# plot_scan_positions(pos_real, init_pos=pos, dot_scale=0.1, show_arrow=False)
print(f"First 5 positions of pos_ang_xy (Ang) = {pos_ang_xy[:5]}, this is for abTEM\n")
print(f"First 5 positions of pos_px_yx (px) = {pos_px_yx[:5]}, this is for reconstruction packages\n")
print(f"First 5 positions of crop_pos (px) = {crop_pos[:5]}, this is for reconstruction packages\n")
print(f"First 5 positions of probe_pos_shifts (px) = {probe_pos_shifts[:5]}, this is for reconstruction packages")

In [ ]:
# Prepare torch tensors
dx = torch.tensor(dx, device='cuda')
dz = torch.tensor(vertical_sampling, device='cuda')
lambd = torch.tensor(get_EM_constants(energy/1e3, 'wavelength'), device='cuda')
crop_pos = torch.tensor(crop_pos, device='cuda', dtype=torch.int32)
probe_pos_shifts = torch.tensor(probe_pos_shifts, device='cuda', dtype=torch.float32)
objp = torch.tensor(objp, device='cuda', dtype=torch.float32)

In [ ]:
# Initialize probes tensor
probe = torch.tensor(probe[None,], device='cuda', dtype=torch.complex64)
Npy, Npx = probe.shape[-2:]

# Grids for shifting probes and obj_ROI selection
rpy_grid, rpx_grid = torch.meshgrid(torch.arange(Npy, dtype=torch.int32, device='cuda'), 
                                    torch.arange(Npx, dtype=torch.int32, device='cuda'), indexing='ij') # real space grid for probe in y and x directions

shift_probes_grid = torch.stack([rpy_grid/Npy, rpx_grid/Npx], dim=0)
probes = imshift_batch(probe, probe_pos_shifts[indices], shift_probes_grid)

In [ ]:
# Initialize basic propagator
H = near_field_evolution_torch(probes.shape[-2:], dx, dz, lambd)

# Grids for propagator tilt (the units are different from the above grid)
Npy = Npx = Npix
dk = 1/(dx*Npix)
ky = fftfreq(Npy, 1 / Npy, device='cuda') * dk # dk in 1/Ang
kx = fftfreq(Npx, 1 / Npx, device='cuda') * dk
Kyt, Kxt = torch.meshgrid(ky, kx, indexing='ij')

tilt_y  = tilts[:,0,None,None] / 1e3 #mrad, tilts_y = (num_tilts,Y,X)
tilt_x  = tilts[:,1,None,None] / 1e3

# Add tilt to the basic propagator
phase_shift = 2 * torch.pi * dz * (Kyt * torch.tan(tilt_y) + Kxt * torch.tan(tilt_x))
propagators = H * torch.exp(1j*phase_shift)

In [ ]:
# Initialize tmask for thickness optimization
tmask = scaled_sigmoid(length=objp.shape[-3], offset = thicknesses/vertical_sampling, scale=1)

In [ ]:
# Get object patches
obj_ROI_grid_y = rpy_grid[None,:,:] + crop_pos[indices, None, None, 0]
obj_ROI_grid_x = rpx_grid[None,:,:] + crop_pos[indices, None, None, 1]

objp_patches = objp[:,:,obj_ROI_grid_y,obj_ROI_grid_x].permute(2,0,1,3,4)

In [ ]:
# Forward pass to generate the diffraction pattern
dp_fwd = multislice_forward_model_vec_all(probes, propagators, tmask, objp_patches, eps=1e-10)

## Visualization of the forward diffraction pattern

In [ ]:
dp_pow = 0.5

dp_fwd_np = dp_fwd[0,0,0].detach().cpu().numpy()

diff = dp_fwd_np**dp_pow - meas**dp_pow

fig, axs = plt.subplots(1,3, figsize=(16,5))
plt.suptitle(f"DP pow = {dp_pow}", y=0.90)
im0 = axs[0].imshow(meas**dp_pow)
im1 = axs[1].imshow(dp_fwd_np**dp_pow)
im2 = axs[2].imshow(diff, cmap='seismic', vmin=-3*diff.std(), vmax=3*diff.std())

axs[0].set_title('Resampled abTEM PACBED')
axs[1].set_title(f'Forward PACBED b={batch_size}')
axs[2].set_title('Difference (forward - abTEM)')

fig.colorbar(im0, shrink=0.6)
fig.colorbar(im1, shrink=0.6)
fig.colorbar(im2, shrink=0.6)
plt.show()